# NBHD-GeoJSON

In [2]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [36]:
from ipywidgets import Widget
import pandas as pd
import celldega as dega
import spatialdata as sd
import scanpy as sc
from spatialdata_io import xenium

In [4]:
base_path = 'https://raw.githubusercontent.com/broadinstitute/celldega_Xenium_human_Pancreas_FFPE/main/Landscape_Xenium_V1_human_Pancreas_FFPE_outs_webp/'
xenium_path = 'data/raw/Xenium_V1_human_Pancreas_FFPE_outs/'
zarr_path = 'data/raw/Xenium_V1_human_Pancreas_FFPE_outs.zarr'

In [5]:
# # load raw data to sdata and save as zarr
# sdata = xenium(xenium_path)
# sdata.write(zarr_path)

In [6]:
cluster = pd.read_parquet(base_path + 'cell_clusters/cluster.parquet')

In [7]:
sdata = sd.read_zarr(zarr_path)
sdata

/Users/whuan/miniconda3/envs/celldega_env/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/whuan/miniconda3/envs/celldega_env/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/whuan/miniconda3/envs/celldega_env/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/whuan/miniconda3/envs/celldega_env/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/whuan/miniconda3/envs/celldega_env/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argum

SpatialData object, with associated Zarr store: /Users/whuan/dev/celldega/notebooks/data/raw/Xenium_V1_human_Pancreas_FFPE_outs.zarr
├── Images
│     └── 'morphology_focus': DataTree[cyx] (5, 13770, 34155), (5, 6885, 17077), (5, 3442, 8538), (5, 1721, 4269), (5, 860, 2134)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (13770, 34155), (6885, 17077), (3442, 8538), (1721, 4269), (860, 2134)
│     └── 'nucleus_labels': DataTree[yx] (13770, 34155), (6885, 17077), (3442, 8538), (1721, 4269), (860, 2134)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 11) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (140702, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (140702, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (136531, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (140702, 377)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), cell_labels (Labels), nucleus_label

In [8]:
adata = sdata.tables["table"]
adata.obs.set_index('cell_id', inplace=True)
adata

AnnData object with n_obs × n_vars = 140702 × 377
    obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region', 'z_level', 'nucleus_count', 'cell_labels'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'

In [9]:
adata.obs['cluster'] = cluster

### Load Data

In [10]:
meta_cluster = pd.read_parquet(base_path + 'cell_clusters/meta_cluster.parquet')
meta_cluster.head()

,color,count
1,#1f77b4,17949
2,#ff7f0e,15781
3,#2ca02c,14415
4,#d62728,11840
5,#9467bd,9526


### Calculate Alpha Shape Neighborhoods

In [31]:
alphas=[20, 50]
gdf_alpha = dega.nbhd.alpha_shape_cell_clusters(
    adata,
    cat='cluster',
    alphas=alphas,
    meta_cluster=meta_cluster
)
gdf_alpha = gdf_alpha[gdf_alpha['inv_alpha'] == 50]

In [29]:
gdf_alpha = gdf_alpha.drop(columns=['area'])

In [32]:

display(gdf_alpha.head(5))

Widget.close_all()
base_url = base_path.rstrip('/')
landscape = dega.viz.Landscape(
    technology='Xenium',
    height=500,
    base_url = base_url,
    nbhd=gdf_alpha,
)
landscape

,name,cat,geometry,inv_alpha,color,area
1_50,1_50,1,"MULTIPOLYGON (((3884.790 66.240, 3843.930 29.8...",50.0,#1f77b4,1.163024e+07
5_50,5_50,5,"MULTIPOLYGON (((775.170 1322.400, 787.630 1374...",50.0,#9467bd,7.615571e+06
4_50,4_50,4,"MULTIPOLYGON (((363.400 2592.870, 409.480 2637...",50.0,#d62728,6.415101e+06
7_50,7_50,7,"MULTIPOLYGON (((171.520 2737.010, 142.620 2718...",50.0,#e377c2,5.761309e+06
2_50,2_50,2,"MULTIPOLYGON (((679.810 1306.350, 693.330 1323...",50.0,#ff7f0e,5.545825e+06


Landscape(base_url='https://raw.githubusercontent.com/broadinstitute/celldega_Xenium_human_Pancreas_FFPE/main/…

In [34]:
gdf_cell = dega.nbhd._get_gdf_cell(adata)
# gdf_micron = gpd.read_parquet(f'{path_landscape_files}/spatial_regions/sketched_regions_micron.parquet')
# gdf_roi = gdf_micron.loc[gdf_micron['roi'] == 'region_1']
# gdf_nbhd = dega.nbhd.calc_grad_nbhd_from_roi(gdf_roi, gdf_cell, 200)
# gdf_nbhd['name'] = gdf_nbhd['band']

KeyError: 'leiden'

In [33]:
adata

AnnData object with n_obs × n_vars = 140702 × 377
    obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region', 'z_level', 'nucleus_count', 'cell_labels', 'cluster', 'geometry'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'

In [37]:
# dataset_name = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'
dataset_name = 'Xenium_V1_human_Pancreas_FFPE_outs'
tech = 'Xenium'
DATA_DIR = f'data/raw'
data_dir = f'{DATA_DIR}/{dataset_name}'
path_landscape_files = f'data/landscape_files/{dataset_name}'



# from spatialdata_io import xenium
# import spatialdata as sd
# import os
# # Ingest xenium data raw output folder using spatialdata-io
# sdata = xenium(data_dir)

# # Write sdata to a zarr file
# zarr_path = f"{data_dir}.zarr"

# # Check if the zarr file already exists
# if os.path.exists(zarr_path):
#     print(f"The file {zarr_path} already exists.")
# else:
#     # If the file does not exist, write the data
#     sdata.write(zarr_path)
#     print(f"Data written to {zarr_path} successfully.")

# # Read zarr file using spatialdata
# sdata = sd.read_zarr(zarr_path)

# # Create anndata from sdata.tables['table]
# adata = sdata.tables["table"]
# adata.write_h5ad(f'{data_dir}.h5ad')


# Load h5ad file
adata_raw = sc.read_h5ad(f'{data_dir}.h5ad')
adata_raw.obs.set_index('cell_id', inplace=True)

In [38]:
adata_raw

AnnData object with n_obs × n_vars = 122678 × 377
    obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region', 'z_level', 'nucleus_count', 'cell_labels', 'n_counts', 'leiden'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells'
    uns: 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'spatialdata_attrs', 'umap'
    obsm: 'X_pca', 'X_umap', 'spatial'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'